# 10 – Ratings

Esplorazione e data cleaning del dataset `ratings.csv`.

| Colonna | Descrizione |
|---|---|
| `username` | Nome utente MAL |
| `anime_id` | ID dell'anime su MAL |
| `status` | Stato di visione |
| `score` | Voto dell'utente |
| `is_rewatching` | Indica se l'utente sta rivedendo l'anime |
| `num_watched_episodes` | Numero di episodi guardati |

## 1. Import e caricamento dati

Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [ ]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_rt = pd.read_csv('../datasets/ratings.csv')
print(f'Shape: {df_rt.shape}')
df_rt.info(show_counts=True)
df_rt.head()

**Osservazioni iniziali:**
- Il dataset contiene 124,298,357 di righe e 6 colonne.
- `username` presenta 7 righe nulle.
- `is_rewatching` contiene 3,797,321 valori null che controlliamo successivamente per verificare se si tratta di un errore o meno.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

La dimensione del dataset (~124M righe) rende la verifica tramite `df.duplicated()` proibitiva in termini di memoria. Usiamo `pd.util.hash_pandas_object()` che calcola un hash `uint64` per ogni riga e cerca duplicati su quella serie, evitando di confrontare le righe intere.

In [ ]:
n_originale = len(df_rt)

hashes = pd.util.hash_pandas_object(df_rt, index=False)
n_dup = hashes.duplicated(keep=False).sum()
print(f'Duplicati esatti sull\'intero dataset: {n_dup:,}')
if n_dup == 0:
    print('→ Nessun duplicato esatto, nessuna operazione richiesta.')
else:
    print('→ Presenza di duplicati: rimozione necessaria.')
    df_rt = df_rt[~hashes.duplicated(keep='first')].reset_index(drop=True)
    print(f'Righe dopo rimozione: {len(df_rt):,}')

Nessun duplicato esatto trovato. Tutte le righe sono già uniche. Il dataset rimane invariato.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `username`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria `username` di `profiles.csv`.

I valori duplicati sono **attesi**: lo stesso utente ha più righe (una per ogni anime in lista).

I controlli rilevanti sono:
- **Valori nulli**: una riga senza username non è collegabile a nessun profilo e va rimossa.
- **Integrità referenziale**: ogni username deve esistere in `profiles_clean.csv`.

Usiamo `check_fk` al posto di `analyze`.

In [ ]:
df_rt['username'] = df_rt['username'].str.strip()
df_profiles = pd.read_csv('../datasets_cleaned/profiles_clean.csv', usecols=['username'])

mask_orphan_user = check_fk(df_rt['username'], df_profiles['username'], child_df=df_rt)

print(f'Null in username               : {df_rt["username"].isna().sum()}')
print(f'Duplicati in username (attesi) : {df_rt["username"].duplicated().sum():,}')

**Osservazioni:**

- Sono presenti 7 righe con `username` null. Senza identificatore utente la riga non è collegabile a nessun profilo quindi va rimossa.
- Ci sono 640,164 righe orfane che vanno rimosse.

In [ ]:
#Rimozione delle righe nulle
print(f'Righe con username null: {df_rt["username"].isna().sum()}')
print()

df_rt.dropna(subset=['username'], inplace=True)
print(f'\nRighe dopo rimozione: {len(df_rt):,}')

#Rimozione delle righe orfane
if mask_orphan_user.any():
    n_orfane = mask_orphan_user.sum()
    df_rt = df_rt[~mask_orphan_user].reset_index(drop=True)
    print(f'Righe orfane rimosse : {n_orfane:,}')
    print(f'Righe rimanenti      : {len(df_rt):,}')
else:
    print('Nessuna riga orfana da rimuovere.')

### 2.2 `anime_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria `mal_id` di `details.csv`.

I valori duplicati sono **attesi**: lo stesso anime compare in più righe (uno per utente).

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID deve esistere in `details_clean.csv`.

Usiamo `check_fk` al posto di `analyze`.

In [ ]:
df_details = pd.read_csv('../datasets_cleaned/details_clean.csv', usecols=['mal_id'])

mask_orphan_anime = check_fk(df_rt['anime_id'], df_details['mal_id'], child_df=df_rt)

print(f'Null in anime_id               : {df_rt["anime_id"].isna().sum()}')
print(f'Duplicati in anime_id (attesi) : {df_rt["anime_id"].duplicated().sum():,}')

**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID anime valido.
- **Integrità referenziale**: ci sono 306,225 righe orfane che rimuoviamo.

In [ ]:
if mask_orphan_anime.any():
    n_orfane = mask_orphan_anime.sum()
    df_rt = df_rt[~mask_orphan_anime].reset_index(drop=True)
    print(f'Righe orfane rimosse : {n_orfane:,}')
    print(f'Righe rimanenti      : {len(df_rt):,}')
else:
    print('Nessuna riga orfana da rimuovere.')

### 2.3 `status`

Questa colonna indica lo stato di visione dell'anime da parte dell'utente. Controlliamo l'eventuale presenza di valori nulli e anomali. Usiamo `analyze` per l'ispezione.

In [ ]:
df_rt['status'] = df_rt['status'].str.strip()
analyze(df_rt['status'])

**Osservazioni:**

- Non ci sono righe nulle e i valori duplicati sono attesi.
- 16.690 righe presentano `status = 'unknown'`. Stampiamo un campione per capire se va rimosso.

In [ ]:
print('Campione di 20 righe con status = "unknown":')
df_rt[df_rt['status'] == 'unknown'].sample(n=20, random_state=42)

Si nota che le righe con status `unknown` sono righe che non contengono altri valori utili e vanno rimosse.

In [ ]:
mask_unknown = df_rt['status'] == 'unknown'
print(f'Righe con status=unknown : {mask_unknown.sum():,}')
df_rt = df_rt[~mask_unknown].copy()
print(f'\nRighe dopo rimozione: {len(df_rt):,}')

### 2.4 `score`

Questa colonna contiene il voto assegnato dall'utente all'anime, su scala intera da 0 a 10. Controlliamo la presenza di valori nulli o anomali. Usiamo `analyze` per l'ispezione.

In [ ]:
analyze(df_rt['score'])

**Osservazioni:**
- Nessun null, dtype già `int64`.
- Ci sono 11 valori unici che corrispondono che le valutazioni da 0 a 10.
- Il valore `0` indica che l'utente **non ha assegnato un voto** (~44% delle righe): è un comportamento atteso su MAL in quano un anime può essere in lista senza essere valutato.

**Nessuna pulizia necessaria.**

### 2.5 `is_rewatching`

Questa colonna indica se l'utente sta rivedendo l'anime per la seconda volta (o più). Usiamo `analyze` per l'ispezione.

In [ ]:
analyze(df_rt['is_rewatching'])

**Osservazioni:**

- Ci sono solo 2 valori unici: `1` e `0`, quindi si tratta di un flag booleano.
- Ci sono **3.775.028 valori null**. Verifichiamo se corrispondono a un `status` specifico come `watching`, `dropped`, `plan_to_watch`, etc.
- Dtype `float64` con valori `0.0`, `1.0` e `NaN` va convertito a `boolean`.

In [ ]:
null_rew  = df_rt['is_rewatching'].isna()
false_rew = df_rt['is_rewatching'] == 0.0
true_rew  = df_rt['is_rewatching'] == 1.0

print(f'Righe is_rewatching = NaN   : {null_rew.sum():>10,}')
print(f'Righe is_rewatching = 0.0   : {false_rew.sum():>10,}')
print(f'Righe is_rewatching = 1.0   : {true_rew.sum():>10,}')
print()

print('Distribuzione status per is_rewatching = NaN:')
print(df_rt[null_rew]['status'].value_counts(dropna=False).to_string())
print()
print('Distribuzione status per is_rewatching = 0.0 (campione top 5):')
print(df_rt[false_rew]['status'].value_counts(dropna=False).head().to_string())
print()
print('Distribuzione status per is_rewatching = 1.0:')
print(df_rt[true_rew]['status'].value_counts(dropna=False).to_string())

**Osservazioni sulla correlazione `is_rewatching` ↔ `status`:**

- I null in `is_rewatching` non corrispondono a un singolo `status` ma sono distribuiti tra tutti i valori. Confermano che si tratta di un valore di default assegnato agli anime aggiunti alla lista prima che MAL introducesse il tracking esplicito del re-watching.
- Per `is_rewatching = 0.0`, tutti i status sono attesi.
- Per `is_rewatching = 1.0`:
    - Le righe con status `completed` e `watching` sono coerenti. L'utente sta rivedento qualcosa che ha già finito oppure è nel mezzo del rewatch.
    - Le righe con status `on_hold` e `dropped` sono plausibili. L'utente ha iniziato un rewatch ma poi l'ha messo in pausa o abbandonato durante la revisione.
    - Le righe con status `plan_to_watch` sono inconsistenti. L'utente non può rivedere qualcosa che non ha ancora visto. Sono solo 757 righe e sono trascurabili.

**Nessuna pulizia aggiuntiva necessaria**: i null sono strutturali e vengono mantenuti.Facciamo solo la conversione a `boolean`.

In [ ]:
print('Valori is_rewatching prima della conversione:')
print(df_rt['is_rewatching'].value_counts(dropna=False))

df_rt['is_rewatching'] = df_rt['is_rewatching'].astype('boolean')

print(f'\nis_rewatching dtype : {df_rt["is_rewatching"].dtype}')
print('Valori dopo la conversione:')
print(df_rt['is_rewatching'].value_counts(dropna=False))

### 2.6 `num_watched_episodes`

Questa colonna indica quanti episodi dell'anime l'utente ha guardato al momento del salvataggio dei dati. Usiamo `analyze` per l'ispezione su eventuali valori nulli o anomali.

In [ ]:
analyze(df_rt['num_watched_episodes'])

**Osservazioni:**

- Nessun null, dtype già `int64`.
- Il valore `0` è atteso per entry con `status = 'plan_to_watch'` o per anime non ancora iniziati.
- Nella distribuzione si nota un conteggio anomalo per l'intervallo 62,258 – 65,535. Si tratta di un comportamento di MAL dove i numeri vengono salvati come interi a 16 bit che permettono di sappresentare al massimo 65,536 valori. Essendo una percentuale molto bassa, non influiscono l'analisi e vanno tenute.

**Nessuna pulizia necessaria.**

### 2.7 Chiave composita `(username, anime_id)`

La combinazione `(username, anime_id)` identifica univocamente una riga: ogni utente ha al più un entry per anime.

In [ ]:
n_pk_dup = df_rt.duplicated(subset=['username', 'anime_id'], keep=False).sum()
print(f'Duplicati su (username, anime_id) sull\'intero dataset: {n_pk_dup:,}')

if n_pk_dup == 0:
    print('→ Chiave composita univoca, nessuna operazione richiesta.')
else:
    print('→ Presenza di duplicati: rimozione necessaria.')
    df_rt = df_rt[~df_rt.duplicated(subset=['username', 'anime_id'], keep='first')].reset_index(drop=True)
    print(f'Righe dopo rimozione: {len(df_rt):,}')

Sono stati trovati 12 duplicati che abbiamo rimosso. La chiave composita `(username, anime_id)` adesso è univoca sull'intero dataset. Il dataset è pronto per il salvataggio.

## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effettuiamo il salvataggio del dataset finale.

In [ ]:
print('Riepilogo Dataset Pulito')
print(f'Righe originali      : {n_originale:>15,}')
print(f'Righe dopo cleaning  : {len(df_rt):>15,}')
print(f'Righe rimosse totali : {n_originale - len(df_rt):>15,}')
print()
df_rt.to_csv('../datasets_cleaned/ratings_clean.csv', index=False)
print('Salvato: datasets_cleaned/ratings_clean.csv')